# Week 6 — Data Cleaning, Leakage Fixes, Feature Engineering & Retraining

Continues from `03_baseline_model.ipynb` and `04_model_comparison.ipynb`. This notebook addresses the open TODOs and Slack feedback before trusting the R² numbers, and adds the feature engineering requested in the AVM Data Science Best Practices doc.

- **Leakage columns removed:** `ListPrice`, `OriginalListPrice` (per Aidan's message — listing agents set these using comparables/market info highly correlated with `ClosePrice`, and they don't exist for off-market properties, which is a primary use case). Also removing `DaysOnMarket` and `PurchaseContractDate` (per Ahyo's question — both are only knowable *after* a sale happens, i.e. after `ClosePrice` is effectively determined, so they leak post-outcome information into training).
- **Data quality checks:** duplicates, numeric columns with zero/negative values where that's not physically meaningful (e.g. `LivingArea`, `LotSizeArea`).
- **Feature engineering (new):**
  - Locational: distance to the nearest major CA employment center from `Latitude`/`Longitude`, plus a KMeans spatial cluster ("geohash-style" neighborhood proxy) fit on training data only.
  - Temporal: month/season encoded with sine/cosine so December and January stay close together, and property age at time of sale.
  - Encoding: high-cardinality categoricals are routed to K-fold, leakage-safe target encoding instead of naive one-hot; low-cardinality categoricals still get one-hot encoded.
  - ZIP-level median price-per-sqft, computed on the training window only and joined forward to both train and test.
- **Outlier filtering:** compute 0.5th/99.5th percentile `ClosePrice` thresholds from the *training set only*, then apply those frozen thresholds to both train and test (no leakage of test distribution into the filter).
- **Re-run training** across Linear Regression, Decision Tree, and Random Forest, and across multiple `month` windows (1, 3, 6, 12) to check how sensitive performance is to the training window size.

**Workflow order below** (each step only uses information that would actually be available at that point, so nothing downstream leaks into something upstream):

1. Imports
2. Load data
3. Data quality checks
4. Drop leakage columns
5. Feature engineering — location & temporal (safe to compute before the split; doesn't use `ClosePrice`)
6. Train/test split (time-based)
7. Outlier filtering (thresholds frozen from train)
8. Leakage-safe feature engineering — geo clusters, ZIP price/sqft, target encoding (must happen *after* the split, fit on train only)
9. Prepare data (assemble final numeric/categorical column lists)
10. Metrics + reusable train/eval
11. End-to-end pipeline for a given training window
12. Run for `month=6`
13. Sensitivity check across training windows
14. Feature importance sanity check


> Feature engineering requirements this notebook implements (from the AVM Data Science Best Practices doc):
>
> **Locational Features**
> — Distance to CBD, major employment centers, or top-rated schools computed from latitude/longitude.
> — Neighborhood or ZIP-level aggregates (e.g., median price per sqft in the last 12 months), computed on training data only and joined forward to avoid leakage.
> — Geohash or spatial clustering as a categorical feature when neighborhood labels are noisy or inconsistent.
>
> **Temporal Features**
> — Month/season encoded with sine/cosine transforms to capture cyclical seasonality without an artificial ordinal jump from December to January.
> — Property age at time of sale, not just year built, so the feature reflects condition-relevant aging.
>
> **Categorical Encoding**
> — For high-cardinality categoricals (neighborhood, subtype), avoid naive one-hot encoding blowing up dimensionality; use target encoding computed with cross-validation folds so a category's encoding never sees its own row's label — a common, subtle leakage source.


In [ ]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error


## 1. Load data

In [ ]:
df = pd.read_csv("data/df_cleaned.csv")
print(df.shape)
df.head()


## 2. Data quality checks

Before doing anything else: duplicates, and numeric columns with 0 or negative values in fields where that isn't physically meaningful (a house can't have 0 sqft living area or a negative lot size, for example — those are probably data errors, not legitimate values).

In [ ]:
# Duplicates
n_dupes = df.duplicated().sum()
print(f"Full-row duplicates: {n_dupes}")

# Drop exact duplicate rows
if n_dupes > 0:
    df = df.drop_duplicates()
    print(f"Shape after dropping duplicates: {df.shape}")


In [ ]:
# Numeric columns with 0 or negative values
numeric_cols_check = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

zero_or_neg_summary = {}
for c in numeric_cols_check:
    n_zero = (df[c] == 0).sum()
    n_neg = (df[c] < 0).sum()
    if n_zero > 0 or n_neg > 0:
        zero_or_neg_summary[c] = {"zero_count": n_zero, "negative_count": n_neg}

zero_or_neg_df = pd.DataFrame(zero_or_neg_summary).T.sort_values("negative_count", ascending=False)
zero_or_neg_df


In [ ]:
# Fields where 0/negative are physically implausible and likely data errors.
# Adjust this list after reviewing zero_or_neg_df above -- not every numeric column
# with a zero is necessarily wrong (e.g. a count field can legitimately be 0).
implausible_if_nonpositive = ["LivingArea", "LotSizeArea", "ClosePrice"]

for col in implausible_if_nonpositive:
    if col in df.columns:
        before = len(df)
        df = df[df[col] > 0]
        print(f"{col}: dropped {before - len(df)} rows with value <= 0")


## 3. Drop leakage columns

Per Aidan's Slack message: `ListPrice` and `OriginalListPrice` are set by listing agents using comparables and market conditions highly correlated with `ClosePrice`, and neither exists for off-market properties (one of the model's primary use cases) — so keeping them both leaks the target and breaks the off-market use case.

Per Ahyo's follow-up: `DaysOnMarket` and `PurchaseContractDate` are only knowable *after* a sale occurs (you don't know how long a property was on the market, or when the contract was signed, until the sale has happened) — so both are also post-outcome information and get dropped alongside `ListPrice`/`OriginalListPrice`.

In [ ]:
LEAKAGE_COLS = ["ListPrice", "OriginalListPrice", "DaysOnMarket", "PurchaseContractDate"]

present = [c for c in LEAKAGE_COLS if c in df.columns]
missing = [c for c in LEAKAGE_COLS if c not in df.columns]
print("Dropping:", present)
if missing:
    print("Not found in df (already absent):", missing)

df = df.drop(columns=present)

assert len(df) > 0, "df is empty after dropping leakage columns -- check upstream filters."


## 5. Feature engineering — location & temporal (before the split)

These features only use each row's own `Latitude`/`Longitude`/`Month`/`YearBuilt` — none of them use `ClosePrice` or any aggregate statistic, so it's safe to compute them once on the full `df` before splitting into train/test. (Anything that *does* need to be fit on data, like the ZIP median price/sqft or the geo clusters, is deferred to section 8, after the split, so it can be fit on train only.)

Column names are resolved defensively with `_first_present()` in case the raw schema differs slightly from what's assumed here — check the printed messages below and adjust the candidate lists if a feature gets skipped.


In [ ]:
def _first_present(df, candidates, label):
    """Return the first candidate column name that exists in df, else None (with a warning)."""
    for c in candidates:
        if c in df.columns:
            return c
    print(f"[feature-eng] WARNING: none of {candidates} found for '{label}' -- "
          f"skipping feature(s) that depend on it. Update the candidate list if the real "
          f"column has a different name.")
    return None


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two (lat, lon) points, vectorized over arrays."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


LAT_COL = _first_present(df, ["Latitude", "latitude", "Lat"], "latitude")
LON_COL = _first_present(df, ["Longitude", "longitude", "Long", "Lon"], "longitude")
MONTH_COL = _first_present(df, ["Month"], "sale month")
YEAR_COL = _first_present(df, ["Year"], "sale year")
YEARBUILT_COL = _first_present(df, ["YearBuilt", "YearBuiltEffective", "Year_Built"], "year built")

# --- Locational: distance to nearest major CA employment center ---
# Statewide data spans multiple metros (San Diego, LA/OC, Bay Area, etc.), so instead of a
# single "distance to CBD" we take the minimum distance to a handful of major centers.
MAJOR_CENTERS_CA = {
    "Downtown_LA": (34.0407, -118.2468),
    "Downtown_San_Diego": (32.7157, -117.1611),
    "Downtown_San_Jose": (37.3382, -121.8863),
    "Downtown_Sacramento": (38.5816, -121.4944),
}

if LAT_COL and LON_COL:
    dist_cols = []
    for name, (clat, clon) in MAJOR_CENTERS_CA.items():
        col = f"dist_km_to_{name}"
        df[col] = haversine_km(df[LAT_COL], df[LON_COL], clat, clon)
        dist_cols.append(col)
    df["dist_km_to_nearest_major_center"] = df[dist_cols].min(axis=1)
    df = df.drop(columns=dist_cols)  # keep only the min -- the per-center distances are redundant
    print(f"Added dist_km_to_nearest_major_center from {LAT_COL}/{LON_COL}")
else:
    print("Skipping distance-to-employment-center feature (no lat/long columns found).")

# --- Temporal: cyclical month/season encoding ---
if MONTH_COL:
    df["month_sin"] = np.sin(2 * np.pi * df[MONTH_COL] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df[MONTH_COL] / 12)
    print(f"Added month_sin / month_cos cyclical encoding from {MONTH_COL}")

# --- Temporal: property age at time of sale (not just year built) ---
if YEARBUILT_COL and YEAR_COL:
    df["property_age_at_sale"] = (df[YEAR_COL] - df[YEARBUILT_COL]).clip(lower=0)
    print(f"Added property_age_at_sale from {YEAR_COL} - {YEARBUILT_COL}")

df.head()


## 6. Split (unchanged from `03_baseline_model.ipynb` / `04_model_comparison.ipynb`)

In [ ]:
def split(df, month):
    assert len(df) > 0, "split() received an empty dataframe -- check filters applied before this call."
    df["month-year"] = pd.to_datetime(df["YearMonth"])
    df = df.sort_values("month-year")
    assert df["month-year"].notna().any(), "YearMonth failed to parse -- all values are NaT. Check the column's format/content."
    latest_month = df["month-year"].max()
    print("Test month:", latest_month)

    test_df = df[df["month-year"] == latest_month]

    train_start = latest_month - pd.DateOffset(months=month)
    train_df = df[
        (df["month-year"] < latest_month) &
        (df["month-year"] >= train_start)
    ]

    print("Training period:", train_df["month-year"].min(), "to", train_df["month-year"].max())
    print("Testing period:", test_df["month-year"].min(), "to", test_df["month-year"].max())
    assert len(train_df) > 0, "train_df is empty after the split -- try a larger `month` window or check date range."
    assert len(test_df) > 0, "test_df is empty after the split -- check that YearMonth has a valid latest month."
    return train_df, test_df


## 7. Outlier filtering on `ClosePrice`

Compute the 0.5th and 99.5th percentile thresholds using the **training set only**, then apply those *same, frozen* thresholds to the test set. This avoids leaking any information about the test set's price distribution into the filter, while still knocking out extreme outliers on both sides.

In [ ]:
def filter_outliers(train_df, test_df, target="ClosePrice", lower_q=0.005, upper_q=0.995):
    assert len(train_df) > 0, "filter_outliers() received an empty train_df."
    lower_thresh = train_df[target].quantile(lower_q)
    upper_thresh = train_df[target].quantile(upper_q)
    print(f"Frozen thresholds from train: [{lower_thresh:,.0f}, {upper_thresh:,.0f}]")

    train_before, test_before = len(train_df), len(test_df)

    train_df = train_df[(train_df[target] >= lower_thresh) & (train_df[target] <= upper_thresh)]
    test_df = test_df[(test_df[target] >= lower_thresh) & (test_df[target] <= upper_thresh)]

    print(f"Train: {train_before} -> {len(train_df)} rows")
    print(f"Test:  {test_before} -> {len(test_df)} rows")
    assert len(train_df) > 0, "All training rows removed by outlier filter -- check ClosePrice values/quantiles."
    assert len(test_df) > 0, "All test rows removed by outlier filter -- thresholds from train may not cover the test month's prices."
    return train_df, test_df


## 8. Feature engineering — fit on train only, applied to test

These features all involve fitting something to data (cluster centroids, ZIP medians, target-encoding maps), so unlike section 5 they have to happen *after* the split — otherwise information from the test period would leak into training through the fitted statistic.

- **Geo clusters**: KMeans fit on training `Latitude`/`Longitude` only, used as a "geohash-style" categorical neighborhood proxy.
- **ZIP median price/sqft**: computed from `ClosePrice / LivingArea` on the training rows only, grouped by `PostalCode`, then mapped onto both train and test (unseen ZIPs in test fall back to the train-wide median).
- **K-fold target encoding**: for high-cardinality categorical columns (auto-detected by unique-value count, so this doesn't depend on guessing exact column names like "neighborhood" or "subtype"), each row's encoding is computed out-of-fold on train so a row's own label never leaks into its own encoding, then the full-train mapping is applied to test.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold


def add_geo_clusters(train_df, test_df, lat_col, lon_col, n_clusters=25, random_state=42):
    """Fit KMeans on train lat/long only; label both train and test with the resulting cluster id."""
    if not (lat_col and lon_col):
        return train_df, test_df
    train_df = train_df.copy()
    test_df = test_df.copy()

    coords_train = train_df[[lat_col, lon_col]].dropna()
    if len(coords_train) < n_clusters:
        print("[geo-cluster] WARNING: not enough non-null coordinate rows to fit clusters -- skipping.")
        return train_df, test_df

    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    km.fit(coords_train)

    fallback = coords_train.mean()
    train_df["geo_cluster"] = km.predict(train_df[[lat_col, lon_col]].fillna(fallback)).astype(str)
    test_df["geo_cluster"] = km.predict(test_df[[lat_col, lon_col]].fillna(fallback)).astype(str)
    return train_df, test_df


def add_zip_price_per_sqft(train_df, test_df, zip_col="PostalCode",
                            price_col="ClosePrice", area_col="LivingArea"):
    """Median $/sqft per ZIP, computed on train only, joined forward onto train and test."""
    for col in (zip_col, price_col, area_col):
        if col not in train_df.columns:
            print(f"[zip-price-psf] WARNING: '{col}' not found -- skipping ZIP price/sqft feature.")
            return train_df, test_df

    train_df = train_df.copy()
    test_df = test_df.copy()

    train_psf = train_df[price_col] / train_df[area_col].replace(0, np.nan)
    zip_median = train_psf.groupby(train_df[zip_col]).median()
    global_median = train_psf.median()

    train_df["zip_median_price_per_sqft"] = train_df[zip_col].map(zip_median).fillna(global_median)
    test_df["zip_median_price_per_sqft"] = test_df[zip_col].map(zip_median).fillna(global_median)
    return train_df, test_df


def kfold_target_encode(train_series, train_target, test_series, n_splits=5, smoothing=10, random_state=42):
    """
    Out-of-fold target encoding: each train row's encoding comes from a mean fit on the
    *other* folds only, so a row's own label never leaks into its own encoded value.
    The test set gets encoded using the mapping fit on the full training set.
    Smoothing pulls sparsely-observed categories toward the global mean.
    """
    global_mean = train_target.mean()
    oof = pd.Series(index=train_series.index, dtype=float)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for fit_idx, val_idx in kf.split(train_series):
        fold_cats = train_series.iloc[fit_idx]
        fold_y = train_target.iloc[fit_idx]
        stats = fold_y.groupby(fold_cats).agg(["mean", "count"])
        smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
        oof.iloc[val_idx] = train_series.iloc[val_idx].map(smoothed).values
    oof = oof.fillna(global_mean)

    full_stats = train_target.groupby(train_series).agg(["mean", "count"])
    full_smoothed = (full_stats["mean"] * full_stats["count"] + global_mean * smoothing) / (full_stats["count"] + smoothing)
    test_encoded = test_series.map(full_smoothed).fillna(global_mean)

    return oof, test_encoded


def split_categoricals_by_cardinality(df, categorical_cols, threshold=15):
    """Route high-cardinality categoricals to target encoding, leave the rest for one-hot."""
    high_card = [c for c in categorical_cols if df[c].nunique() > threshold]
    low_card = [c for c in categorical_cols if c not in high_card]
    return high_card, low_card


## 9. Prepare data (now target-encodes high-cardinality categoricals)

In [ ]:
def prepare_data(train_df, test_df, target="ClosePrice", cardinality_threshold=15):
    drop_cols = [target, "month-year", "YearMonth"]
    X_train = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
    y_train = train_df[target]

    X_test = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
    y_test = test_df[target]

    categorical_cols_all = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    high_card_cols, low_card_cols = split_categoricals_by_cardinality(
        X_train, categorical_cols_all, threshold=cardinality_threshold
    )

    # K-fold target-encode high-cardinality categoricals (fit on train only, applied to test)
    for col in high_card_cols:
        oof_enc, test_enc = kfold_target_encode(
            X_train[col].astype(str), y_train, X_test[col].astype(str)
        )
        X_train[f"{col}_target_enc"] = oof_enc.values
        X_test[f"{col}_target_enc"] = test_enc.values

    if high_card_cols:
        print(f"Target-encoded high-cardinality columns (>{cardinality_threshold} categories): {high_card_cols}")
    X_train = X_train.drop(columns=high_card_cols)
    X_test = X_test.drop(columns=high_card_cols)

    numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_cols = low_card_cols
    if categorical_cols:
        print(f"One-hot encoding low-cardinality columns: {categorical_cols}")

    return X_train, y_train, X_test, y_test, numeric_cols, categorical_cols


## 10. Metrics + reusable train/eval (from `04_model_comparison.ipynb`)

In [ ]:
def compute_metrics(y_test, y_pred):
    return {
        "r2": r2_score(y_test, y_pred),
        "mae": mean_absolute_error(y_test, y_pred),
        "mape": mean_absolute_percentage_error(y_test, y_pred),
        "mdape": np.median(np.abs((y_test - y_pred) / y_test))
    }


def train_and_evaluate(model, model_name, X_train, y_train, X_test, y_test, numeric_cols, categorical_cols):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ]
    )

    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", model),
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    metrics = compute_metrics(y_test, y_pred)
    metrics["model"] = model_name

    print(f"{model_name:20s} | R\u00b2: {metrics['r2']:.4f} | MAE: {metrics['mae']:,.0f} | "
          f"MAPE: {metrics['mape']:.2%} | MdAPE: {metrics['mdape']:.2%}")

    return metrics, pipeline


## 11. End-to-end run for a given training window

Wraps split -> outlier filtering (frozen thresholds from train) -> leakage-safe feature engineering (geo clusters, ZIP price/sqft, target encoding, all fit on train only) -> prepare_data -> train all three models, and returns a comparison table.

In [ ]:
def run_pipeline(df, month):
    print(f"\n{'='*60}\nmonth = {month}\n{'='*60}")
    train_df, test_df = split(df.copy(), month)
    train_df, test_df = filter_outliers(train_df, test_df)
    train_df, test_df = add_geo_clusters(train_df, test_df, LAT_COL, LON_COL)
    train_df, test_df = add_zip_price_per_sqft(train_df, test_df)
    X_train, y_train, X_test, y_test, numeric_cols, categorical_cols = prepare_data(train_df, test_df)

    models = {
        "Linear Regression": LinearRegression(),
        "Decision Tree": DecisionTreeRegressor(max_depth=6, min_samples_leaf=10, random_state=42),
        "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1),
    }

    results = []
    fitted_pipelines = {}
    for name, model in models.items():
        metrics, pipeline = train_and_evaluate(
            model, name, X_train, y_train, X_test, y_test, numeric_cols, categorical_cols
        )
        metrics["month"] = month
        results.append(metrics)
        fitted_pipelines[name] = pipeline

    comparison_df = pd.DataFrame(results)[["month", "model", "r2", "mae", "mape", "mdape"]]
    comparison_df = comparison_df.sort_values("r2", ascending=False).reset_index(drop=True)
    return comparison_df, fitted_pipelines


## 12. Run for `month=6`

In [ ]:
comparison_df, fitted_pipelines = run_pipeline(df, month=6)
comparison_df


## 13. Sensitivity check across training windows

Per the baseline TODO: test different training window sizes (`month=1,3,6,12`) to see whether performance -- and the ranking between models -- holds up, or whether it's an artifact of one particular window.

In [ ]:
all_results = []
for m in [1, 3, 6, 12]:
    cdf, _ = run_pipeline(df, month=m)
    all_results.append(cdf)

all_results_df = pd.concat(all_results, ignore_index=True)
all_results_df.sort_values(["month", "r2"], ascending=[True, False]).reset_index(drop=True)


## 14. Feature importance sanity check (tree-based models, month=6 run)

Confirms the leakage columns no longer show up, and gives a read on what the models are actually keying off of now -- including the new location/temporal/encoded features.

In [ ]:
def get_feature_names(pipeline):
    return pipeline.named_steps["preprocessing"].get_feature_names_out()

for name in ["Decision Tree", "Random Forest"]:
    pipeline = fitted_pipelines[name]
    feature_names = get_feature_names(pipeline)
    importances = pipeline.named_steps["model"].feature_importances_

    top_features = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(10)
    )
    print(f"\nTop 10 features -- {name}")
    print(top_features.to_string(index=False))
